# test 알약 클래스 빈도 EDA (2차 실험 val 구성)

## EDA의 목적
- 2차 실험의 val을 test와 닮게 만들기에 앞서, test 842장에 나오는 알약을 하나씩 잘라 클래스별 개수를 추정한다.
    - 1차 val(Kaggle 47장)은 클래스 30개만 있고 test 전용 알약이 하나도 없어서, val mAP(0.96~0.98)와 Kaggle 점수(0.39~0.42)가 크게 달랐다.
- 본 EDA를 통해 대회 클래스(채점하고 제출하는 클래스)를 확정하고, 2차 val의 클래스별 목표 객체 수를 정할 test 빈도를 만든다.
    - 00 5-3에서 찾은 test 전용 알약 18종 밖의 알약이 test에 있는지 확인한다.
- 각 섹션의 분석 결과를 crop 규칙, 놓친 알약, 자동 클래스, 검수 결과, 대회 클래스에 대한 결정으로 연결하고, 5번 EDA 결론에서 결정 사항 표로 정리한다.
- 결과는 검수표 `02_raewoo_test_crops.csv`(test 알약 하나가 한 줄)이다.
    - `scripts/make_dataset_lists.py`가 이 표의 `final_category_id`로 대회 클래스와 클래스별 test 객체 수를 센다.
    - 추정한 클래스는 val 구성에만 쓰고, 학습 라벨로는 쓰지 않는다.
- 입력
    - RT-DETR 1차 test 예측: 재원님 `exp/rt-detr` 브랜치의 커밋 `ca00db8`에 있는 `submission_rtdetr-l_e100_s1_fix.csv` (Kaggle 0.4176)
    - 00 test 기록표 `00_raewoo_test_review.csv`: 00 5-2에서 test 사진을 보고 적은, train에 없는 알약 453개
    - 정제한 학습 데이터(`uv run scripts/prepare_train_data.py`): 기준 crop과 각인 필드

## EDA 목차
> 각 분석 결과 아래에는 **인사이트 / 결론** 마크다운 셀을 두어, 분석 결과와 그에 따른 결정을 기록한다.

0. 사전 설정
    - 경로, 도우미 함수를 정의하고, RT-DETR 예측, 00 기록표, 학습 데이터를 읽는다.
1. test 알약 crop 만들기
    - **사진당 알약 하나에 crop 하나**
        - 클래스를 가리지 않고 겹친 예측 box를 합치는 규칙과 최소 score별로 사진당 crop 수를 비교한다.
            - 목적: crop 규칙을 정한다.
    - **놓친 알약**
        - crop이 4개보다 적은 사진의 빈 사분면을 모두 보고, 알약이 보이면 box를 그려 넣는다.
            - 목적: 모델이 놓친 알약도 test 빈도에 넣는다.
2. 자동 클래스
    - **00 기록표와 잇기**
        - crop 중심의 사분면으로 00 기록표의 줄과 잇고, 각인으로 K-코드를 찾는다. (01 5-1 방식)
    - **자동 클래스 정하기**
        - 기록표 각인(review_table), train과 같은 조합의 라벨(known_combo), RT-DETR 예측(prediction) 순서로 자동 클래스를 정한다.
    - **예측 클래스의 정확도**
        - train과 같은 조합인 test 사진의 crop으로 RT-DETR 예측 클래스가 얼마나 맞는지 잰다.
            - 목적: 검수에서 어떤 crop을 집중해서 볼지 정한다.
3. 클래스별 crop 검수
    - **클래스별 crop 시트**
        - 자동 클래스마다 학습 데이터 crop(기준)과 test crop을 한 시트에 놓고, 기준과 다른 알약을 찾는다.
    - **검수 결과**
        - 다른 알약은 각인을 학습 데이터 각인 필드에서 찾고(01 5-1), 사진을 나란히 놓고 확인한다(01 5-2). 못 찾으면 학습할 수 없는 알약으로 센다.
            - 목적: crop마다 최종 클래스를 정한다.
4. 결과
    - **클래스별 test 객체 수**
        - 최종 클래스로 클래스별 test 객체 수를 센다.
    - **대회 클래스 확정**
        - 00 5-3의 18종 밖에서 새로 찾은 클래스, test에 나오지 않는 Kaggle 클래스, 학습할 수 없는 알약을 정리하고 검수표를 저장한다.
            - 목적: 대회 클래스와 2차 val의 test 빈도를 확정한다.
5. EDA 결론
    - 1~4번의 결과를 crop 규칙, 놓친 알약, 자동 클래스, 검수, 대회 클래스 항목별 결정 사항 표로 정리한다.

## 0. 사전 설정

In [ ]:
import collections
import csv
import json
import subprocess
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont

# 학습 데이터 읽기, 파일명 해석, 대회 클래스는 공용 모듈의 함수를 쓴다
from pill_detection.data import combo_name
from pill_detection.datasets import IMAGE_DIRS, TEST_ONLY_CLASSES, load_images

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")

In [ ]:
# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
rng = np.random.default_rng(SEED)

# 노트북을 어디서 실행해도 레포 최상위(pyproject.toml이 있는 폴더)를 기준으로 경로를 잡는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATA_DIR = PROJECT_ROOT / 'data'
TEST_IMG_DIR = DATA_DIR / 'raw' / 'test_images'
EDA_DIR = PROJECT_ROOT / 'notebooks' / 'eda'
OUT_CSV = EDA_DIR / '02_raewoo_test_crops.csv'  # 이 노트북의 결과 (scripts/make_dataset_lists.py가 읽는다)
# 그림, crop 시트, 계산 결과 저장 경로 (runs/는 git에서 제외됨)
FIG_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'figures_02'
SHEET_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'crops_02'
CACHE_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'cache'
for folder in (FIG_DIR, SHEET_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# RT-DETR 1차 test 예측 (재원님 exp/rt-detr 브랜치의 커밋 ca00db8, Kaggle 0.4176, conf 0.01, 사진당 최대 300개)
# 브랜치의 최신 커밋에서는 파일이 지워졌으므로 커밋을 지정해 꺼낸다
PRED_COMMIT, PRED_PATH = 'ca00db8', 'notebooks/modeling/submission_rtdetr-l_e100_s1_fix.csv'
PRED_CSV = CACHE_DIR / 'rtdetr_exp1_test_predictions.csv'

IMG_W, IMG_H = 976, 1280  # 대회 사진 크기 (너비, 높이)
QUADRANTS = ['왼쪽 위', '오른쪽 위', '왼쪽 아래', '오른쪽 아래']  # 00 기록표의 position 값

# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트 사용 (macOS, Windows, Linux 순)
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if font_name in installed_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호가 깨지는 문제 방지


def save_fig(fig, name):
    """그림을 FIG_DIR에 png 파일로 저장한다."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def show_table(header, rows):
    """열 이름 리스트와 행 리스트를 마크다운 표로 출력한다."""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + ' --- |' * len(header)]
    # 값 속 |(예: 분할선 각인 K|50)가 칸 구분자로 읽히지 않게 \|로 바꾼다
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def box_iou(box_a, box_b):
    """두 xywh box의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 완전히 같음)를 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    inter = inter_w * inter_h
    return inter / (box_a[2] * box_a[3] + box_b[2] * box_b[3] - inter)


def box_cover(box_a, box_b):
    """작은 box가 큰 box 안에 들어간 비율(겹친 넓이 ÷ 작은 box 넓이, 1이면 완전히 들어감)을 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    return inter_w * inter_h / max(min(box_a[2] * box_a[3], box_b[2] * box_b[3]), 1e-9)


def on_border(box, margin=3):
    """box가 사진 가장자리(margin 픽셀 안)에 닿는지. 알약은 사진 가장자리에 걸치지 않는다. (01 3-4)"""
    x, y, w, h = box
    return x <= margin or y <= margin or x + w >= IMG_W - margin or y + h >= IMG_H - margin


def quadrant(box):
    """box 중심이 있는 사분면. 00 기록표의 position과 같은 이름이다."""
    cx, cy = box[0] + box[2] / 2, box[1] + box[3] / 2
    return f"{'왼쪽' if cx < IMG_W / 2 else '오른쪽'} {'위' if cy < IMG_H / 2 else '아래'}"


def crop_square(image, box, pad=0.1, size=128):
    """box 중심에서 긴 변보다 pad만큼 큰 정사각형을 잘라 size×size로 줄인다. 알약 모양이 늘어나지 않는다."""
    x, y, w, h = box
    side = max(w, h) * (1 + pad)
    cx, cy = x + w / 2, y + h / 2
    return image.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size))


print(f"레포 최상위  : {PROJECT_ROOT}")
print(f"그래프 폰트  : {plt.rcParams['font.family'][0]}")
print(f"그림 저장    : {FIG_DIR}")

- RT-DETR 1차 test 예측, 00 test 기록표, 정제한 학습 데이터를 읽는다.
    - 예측 CSV는 처음 실행할 때 git으로 커밋 `ca00db8`에서 꺼내 `runs/eda_raewoo/cache/`에 둔다. 커밋이 없으면 `git fetch origin exp/rt-detr`를 먼저 실행한다.
    - 학습 데이터는 `pill_detection.datasets.load_images`로 읽는다. 2차 실험 목록을 만드는 코드와 같은 라벨이다.
    - `class_info`: 클래스마다 JSON 하나의 이미지 정보(각인, 색, 모양)이다. 3-1 시트 제목과 2-1의 각인 대조에 쓴다.

In [ ]:
if not PRED_CSV.exists():
    # git이 커밋의 파일을 꺼낸다. 커밋이 없으면 git fetch origin exp/rt-detr를 먼저 실행한다
    PRED_CSV.write_bytes(subprocess.run(['git', 'show', f'{PRED_COMMIT}:{PRED_PATH}'], cwd=PROJECT_ROOT,
                                        capture_output=True, check=True).stdout)
predictions = collections.defaultdict(list)  # test 파일 → [(score, 예측 클래스, xywh box)]
with open(PRED_CSV, encoding='utf-8') as f:
    for row in csv.DictReader(f):
        box = [float(row[key]) for key in ('bbox_x', 'bbox_y', 'bbox_w', 'bbox_h')]
        predictions[f"{row['image_id']}.png"].append((float(row['score']), int(row['category_id']), box))
test_files = sorted((p.name for p in TEST_IMG_DIR.glob('*.png')), key=lambda name: int(name.split('.')[0]))
assert set(predictions) == set(test_files), '예측 CSV의 사진과 test 사진 목록이 다르다'

with open(EDA_DIR / '00_raewoo_test_review.csv', encoding='utf-8') as f:
    review_rows = list(csv.DictReader(f))

train_images, class_names = load_images(DATA_DIR)  # 정제한 학습 데이터 (Kaggle 사본 + AI Hub 사본)
kaggle_classes = {c for image in train_images.values() if image['source'] == 'kaggle' for c, _, _ in image['objects']}
class_info = {}  # K-코드 → 그 클래스 JSON 하나의 images[0] (각인, 색, 모양)
for source, folder in (('kaggle', 'processed/train_annotations'), ('aihub', 'processed/ai_hub/train_annotations')):
    for path in sorted((DATA_DIR / folder).glob('*/K-*/*.json')):
        code = int(path.parent.name.removeprefix('K-'))
        if code not in class_info:
            class_info[code] = json.loads(path.read_text(encoding='utf-8'))['images'][0]
show_table(['항목', '값'], [
    ['test 사진', f'{len(test_files):,}장'],
    ['RT-DETR 예측 box', f'{sum(map(len, predictions.values())):,}개 (사진당 최대 {max(map(len, predictions.values()))}개)'],
    ['00 test 기록표', f'{len(review_rows)}줄 (test 사진 {len({r["test_file"] for r in review_rows})}장, 각인 {len({r["imprint"] for r in review_rows})}종)'],
    ['학습 데이터', f'{len(train_images):,}장, 클래스 {len(class_names)}개 (Kaggle train 클래스 {len(kaggle_classes)}개)'],
])

## 1. test 알약 crop 만들기
- RT-DETR 1차 예측 box로 test 사진의 알약을 하나씩 자른다.
    - 1차 모델은 Kaggle train 클래스 56개만 배웠다. 그래서 test 전용 알약에도 Kaggle 클래스 중 하나를 붙인다. 여기서는 box 위치만 믿고, 클래스는 2번에서 따로 정한다.
- 대회 사진에는 알약이 3~4개 있다. (Kaggle train 파일명의 K-코드 수: 3개 157장, 4개 75장)

### 1-1. 사진당 알약 하나에 crop 하나
- 예측 box를 score가 높은 순서로 남기고, 이미 남긴 box와 겹치는 box는 클래스와 상관없이 버린다.
    - 클래스 구분 없는 NMS(겹치는 box 중 score가 가장 높은 것만 남기는 방법): IoU(겹친 넓이 ÷ 합친 넓이) 0.5 이상이면 버린다.
    - 들어간 비율(겹친 넓이 ÷ 작은 box 넓이) 0.6 이상도 버린다. 큰 box 안의 작은 조각 box를 지운다.
    - 가장자리 box 제외: 사진 가장자리 3픽셀 안에 닿는 box를 뺀다. 알약은 사진 가장자리에 걸치지 않는다. (01 3-4)
- 규칙과 최소 score별로 crop이 2개, 3개, 4개, 5개 이상인 사진 수를 센다.

In [ ]:
def merge_boxes(preds, min_score, iou_max=0.5, cover_max=None, skip_border=False):
    """클래스를 가리지 않고 score가 높은 box부터 남긴다. 남긴 box와 IoU가 iou_max 이상이거나, 들어간 비율이 cover_max 이상이면 버린다."""
    kept = []
    for score, code, box in sorted(preds, key=lambda p: -p[0]):
        if score < min_score:
            break
        if skip_border and on_border(box):
            continue
        if all(box_iou(box, k[2]) < iou_max and (cover_max is None or box_cover(box, k[2]) < cover_max) for k in kept):
            kept.append((score, code, box))
    return kept


RULES = [('클래스 구분 없는 NMS (IoU 0.5)', {}), ('+ 들어간 비율 0.6 이상도 버림', {'cover_max': 0.6}),
         ('+ 가장자리 box 제외', {'cover_max': 0.6, 'skip_border': True})]
rows = []
for name, kwargs in RULES:
    for min_score in (0.05, 0.1, 0.3):
        counts = collections.Counter(len(merge_boxes(predictions[f], min_score, **kwargs)) for f in test_files)
        rows.append([name, min_score] + [f'{counts[n]}장' if counts[n] else '-' for n in (2, 3, 4)]
                    + [f'{sum(v for n, v in counts.items() if n >= 5)}장'])
show_table(['crop 규칙', '최소 score', '알약 2개', '3개', '4개', '5개 이상'], rows)

**인사이트**
- 클래스 구분 없는 NMS만 쓰면, 최소 score 0.05~0.3 모두 crop이 5개 이상인 사진이 53~156장 남는다.
    - 남는 box는 대부분 사진 가장자리(접시 밖 흰 테두리)에 걸친 작은 box이다. (탐색 중 확인)
- 들어간 비율 0.6을 더해도 5개 이상인 사진은 51~137장으로 거의 줄지 않는다.
- 가장자리 box까지 빼면, 최소 score 0.1과 0.3의 결과가 같다: 알약 2개 6장, 3개 151장, 4개 685장, 5개 이상 0장
    - 0.05에서는 5개 이상인 사진이 2장 생긴다.

**결론**
- crop 규칙은 클래스 구분 없는 NMS(IoU 0.5) + 들어간 비율 0.6 + 가장자리 box 제외, 최소 score 0.1로 정한다. (`MIN_SCORE`)
    - 0.1~0.3에서 결과가 같으므로, 가장 낮은 0.1을 써서 score가 낮은 알약을 덜 놓친다.
- crop이 4개보다 적은 사진 157장은 1-2에서 빈 사분면을 직접 본다.

### 1-2. 놓친 알약
- 1-1에서 정한 규칙으로 만든 crop은 모든 사진에서 사분면 하나에 하나 이하이다. (셀이 확인한다)
    - 그래서 crop이 4개보다 적은 사진의 빈 사분면에 알약이 있으면, 모델이 놓친 알약이다.
- 빈 사분면을 모두 그리고, 알약이 보인 곳은 `MISSED_PILLS`에 적는다.

In [ ]:
MIN_SCORE = 0.1  # 1-1 표를 보고 정한다
crops = {}  # (test 파일, 사분면) → {'box', 'score', 'pred'}
for f in test_files:
    for score, code, box in merge_boxes(predictions[f], MIN_SCORE, cover_max=0.6, skip_border=True):
        key = (f, quadrant(box))
        assert key not in crops, f'{f}의 {key[1]}에 crop이 두 개다'
        crops[key] = {'box': box, 'score': score, 'pred': code}
empty = [(f, q) for f in test_files if sum((f, q) in crops for q in QUADRANTS) < 4 for q in QUADRANTS if (f, q) not in crops]

QUAD_ORIGIN = {'왼쪽 위': (0, 0), '오른쪽 위': (IMG_W // 2, 0), '왼쪽 아래': (0, IMG_H // 2), '오른쪽 아래': (IMG_W // 2, IMG_H // 2)}
cols = 14
fig, axes = plt.subplots(int(np.ceil(len(empty) / cols)), cols, figsize=(cols * 1.6, np.ceil(len(empty) / cols) * 2.2),
                         layout='constrained')
for ax, (f, q) in zip(axes.flat, empty):
    x0, y0 = QUAD_ORIGIN[q]
    ax.imshow(Image.open(TEST_IMG_DIR / f).crop((x0, y0, x0 + IMG_W // 2, y0 + IMG_H // 2)).reduce(4))
    ax.set_title(f"{f.split('.')[0]} {q}", fontsize=8)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(f'crop이 4개보다 적은 test 사진 {len({f for f, _ in empty})}장의 빈 사분면 {len(empty)}곳 (사분면 하나 = 사진의 1/4)\n'
             '알약이 보이면 RT-DETR이 놓친 알약이다', fontsize=13)
save_fig(fig, '1-2_empty_quadrants')
plt.show()

**인사이트**
- 빈 사분면 163곳 중 24곳은 사분면 가운데에 알약이 통째로 보인다. RT-DETR이 놓친 알약이다.
    - 대부분 큰 타원형이나 장방형 알약이다: 분홍 5010 5개, 하양 DGTH 4개, 분홍 515 4개, 파랑 81 4개, OMACOR 연질캡슐 2개, 십자 분할선 원형 2개, 갈색 연질캡슐 2개, 빨강 타원형 1개(1213.png, 00 기록표의 MG)
- 나머지 139곳은 비어 있거나, 옆 사분면 알약의 가장자리만 보인다.
    - 가장자리에 조각이 크게 보인 8장(308, 485, 694, 866, 941, 1043, 1316, 1477)은 사진 전체를 그려 확인했다. 조각은 모두 옆 사분면 crop의 알약이다.

- `MISSED_PILLS`의 알약을 crop 목록에 넣는다. box는 사분면을 좌표 격자와 함께 확대해 눈으로 읽은 대략의 값이다.
    - 배경 밝기가 사진마다 달라서, 배경과 색이 다른 덩어리를 찾는 방법(OpenCV)은 24개 중 절반 가까이에서 box를 찾지 못했다.
    - 이 box는 crop을 만들어 클래스를 보는 데만 쓴다. 학습 라벨이 아니라서 정밀하지 않아도 된다.
- 그려 넣은 box(빨강)가 알약을 감싸는지 확인한다.

In [ ]:
# 1-2 그림에서 사분면 가운데에 알약이 보인 곳 (사진, 사분면) → (xywh box, 보이는 알약)
# box는 사분면을 좌표 격자와 함께 확대해 눈으로 읽은 대략의 값이다. crop을 만들어 클래스를 보는 데만 쓴다
# 사분면 가장자리에만 알약 조각이 보인 곳은 옆 사분면 crop의 알약이라 넣지 않는다 (308, 485, 694, 866, 941, 1043, 1316, 1477에서 확인)
MISSED_PILLS = {
    ('56.png', '왼쪽 아래'): ([0, 600, 485, 475], '노랑 투명 연질캡슐 OMACOR'),
    ('65.png', '왼쪽 아래'): ([10, 645, 290, 325], '분홍 원형, 십자 분할선'),
    ('124.png', '왼쪽 아래'): ([5, 730, 345, 480], '하양 타원형 DGTH'),
    ('125.png', '오른쪽 위'): ([640, 40, 330, 430], '하양 타원형 DGTH'),
    ('265.png', '오른쪽 아래'): ([628, 670, 340, 460], '분홍 타원형 515'),
    ('275.png', '왼쪽 아래'): ([10, 625, 290, 345], '분홍 원형, 십자 분할선'),
    ('380.png', '왼쪽 아래'): ([0, 600, 480, 440], '노랑 투명 연질캡슐 OMACOR'),
    ('419.png', '오른쪽 아래'): ([568, 690, 400, 440], '분홍 타원형 515'),
    ('518.png', '왼쪽 위'): ([5, 30, 305, 460], '파랑 타원형 81'),
    ('519.png', '오른쪽 아래'): ([618, 720, 270, 540], '파랑 타원형 81'),
    ('591.png', '오른쪽 아래'): ([658, 650, 300, 480], '분홍 타원형 515'),
    ('610.png', '왼쪽 위'): ([5, 130, 345, 420], '분홍 장방형 5010'),
    ('611.png', '오른쪽 아래'): ([608, 670, 360, 460], '분홍 장방형 5010'),
    ('612.png', '왼쪽 위'): ([30, 170, 280, 490], '분홍 장방형 5010'),
    ('650.png', '왼쪽 아래'): ([20, 700, 280, 320], '갈색 타원형 연질캡슐'),
    ('766.png', '오른쪽 아래'): ([588, 700, 380, 530], '분홍 장방형 5010'),
    ('870.png', '오른쪽 아래'): ([578, 710, 380, 510], '파랑 타원형 81'),
    ('923.png', '오른쪽 아래'): ([568, 720, 400, 440], '분홍 타원형 515'),
    ('956.png', '오른쪽 위'): ([608, 5, 270, 455], '하양 타원형 DGTH'),
    ('1126.png', '오른쪽 아래'): ([538, 720, 430, 510], '파랑 타원형 81'),
    ('1213.png', '왼쪽 위'): ([140, 30, 240, 450], '빨강 타원형 (00 기록표 MG)'),
    ('1280.png', '오른쪽 위'): ([598, 5, 240, 465], '하양 타원형 DGTH'),
    ('1296.png', '오른쪽 아래'): ([598, 760, 370, 515], '분홍 장방형 5010'),
    ('1433.png', '왼쪽 아래'): ([10, 650, 300, 350], '갈색 타원형 연질캡슐'),
}
for (f, q), (box, _) in MISSED_PILLS.items():
    assert (f, q) not in crops and quadrant(box) == q, f'{f} {q}: 이미 crop이 있거나 box 중심이 그 사분면이 아니다'
    crops[(f, q)] = {'box': [float(v) for v in box], 'score': None, 'pred': None}

cols = 6
fig, axes = plt.subplots(max(1, int(np.ceil(len(MISSED_PILLS) / cols))), cols, figsize=(cols * 3, max(1, np.ceil(len(MISSED_PILLS) / cols)) * 3.8),
                         layout='constrained', squeeze=False)
for ax, (f, q) in zip(axes.flat, MISSED_PILLS):
    image = Image.open(TEST_IMG_DIR / f).convert('RGB')
    draw = ImageDraw.Draw(image)
    for (ff, qq), crop in crops.items():
        if ff == f:
            x, y, w, h = crop['box']
            draw.rectangle([x, y, x + w, y + h], outline='red' if qq == q else 'lime', width=8)
    ax.imshow(image.reduce(4))
    ax.set_title(f'{f} {q}', fontsize=10)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(f'놓친 알약 {len(MISSED_PILLS)}개에 그려 넣은 box (빨강 = 그려 넣은 box, 초록 = RT-DETR crop)', fontsize=13)
save_fig(fig, '1-2_missed_pills')
plt.show()

**인사이트**
- 그려 넣은 box 24개(빨강)가 모두 알약을 감싼다.
- 같은 사진의 다른 알약(초록)은 모두 RT-DETR crop이 있다. 사진마다 놓친 알약은 하나이다.

**결론**
- 놓친 알약 24개를 crop 목록에 넣는다. crop은 3,229개가 되고, 사진마다 3개(139장) 또는 4개(703장)이다. (4-2 저장 셀 기준)
    - 클래스는 2-2(기록표, 조합)와 3-2(각인)에서 정한다.
- 1차 RT-DETR은 큰 타원형 알약을 일부 사진에서 아예 검출하지 못한다. 5번의 추가 실험 후보에 적는다.

## 2. 자동 클래스
- crop마다 근거가 가장 확실한 방법으로 자동 클래스를 정한다. 3번 검수는 자동 클래스로 묶어서 본다.

### 2-1. 00 기록표와 잇기
- 00 기록표는 train에 없는 알약 하나가 한 줄이고, 사진(test_file)과 사분면(position)이 적혀 있다.
- 같은 사진, 같은 사분면의 crop과 잇는다.
- 기록표의 각인을 학습 데이터 각인 필드에서 찾는다. (01 5-1과 같은 방법)
    - 각인 필드(print_front, print_back)와 마크 설명 필드(mark_code_front_anal, mark_code_back_anal)에서 찾는다.
    - 01 5-2에서 다른 약으로 확인한 (각인, K-코드)는 `DIFFERENT_PILLS`로 뺀다.
        - 01은 AI Hub 클래스에서만 찾았다. 여기서는 Kaggle 클래스도 함께 찾으므로, SK가 Kaggle의 K-003483(초록 원형)과도 같다. 00 기록표 메모대로 뺀다.

In [ ]:
def normalize(text):
    """띄어쓰기와 대소문자를 무시하고 비교하도록 각인 글자를 정리한다. (01 5-1)"""
    return str(text or '').replace(' ', '').lower()


IMPRINT_KEYS = ['print_front', 'print_back', 'mark_code_front_anal', 'mark_code_back_anal']  # 각인 필드와 마크 설명 필드
imprint_fields = {c: tuple(normalize(info.get(key)) for key in IMPRINT_KEYS) for c, info in class_info.items()}


def match_imprint(seen):
    """각인 글자를 학습 데이터 클래스의 각인 필드에서 찾는다. 값 전체가 같은 클래스, 없으면 값에 들어 있는 클래스. (01 5-1)"""
    query = normalize(seen.replace('|', '분할선'))
    hits = [c for c, fields in imprint_fields.items() if query in fields]
    if not hits and len(query) >= 2:
        hits = [c for c, fields in imprint_fields.items() if any(query in field for field in fields)]
    return hits


# 각인 글자는 같지만 test 알약과 다른 약으로 확인한 (각인, K-코드). 01 5-2와 00 기록표 메모
DIFFERENT_PILLS = {
    ('Hanmi', 25469): '아모잘탄정은 분홍 타원형이다. test의 Hanmi 알약은 노랑 타원형인 K-031705(낙소졸정)이다 (01 5-2)',
    ('SK', 3483): '기넥신에프정은 초록 원형이다. test의 SK 알약은 노랑 타원형이다 (00 기록표 메모)',
}
code_of_imprint = {}
for seen in sorted({r['imprint'] for r in review_rows}):
    hits = [c for c in match_imprint(seen) if (seen, c) not in DIFFERENT_PILLS]
    assert len(hits) == 1, f'각인 {seen}의 K-코드가 하나가 아니다: {hits}'
    code_of_imprint[seen] = hits[0]
assert sorted(set(code_of_imprint.values())) == sorted(TEST_ONLY_CLASSES), '기록표 각인의 K-코드가 datasets.TEST_ONLY_CLASSES와 다르다'

review_of = {}  # (test 파일, 사분면) → 기록표 줄
for row in review_rows:
    key = (row['test_file'], row['position'])
    assert key not in review_of, f'기록표에 같은 사진, 같은 사분면 줄이 두 개다: {key}'
    review_of[key] = row
linked = [key for key in review_of if key in crops]
show_table(['항목', '줄 수'], [
    ['00 기록표 줄 (train에 없는 알약 하나가 한 줄)', len(review_of)],
    ['같은 사진, 같은 사분면의 crop과 이어진 줄', len(linked)],
    ['이어지지 않은 줄', f"{len(review_of) - len(linked)} ({', '.join(f'{f} {q}' for f, q in review_of if (f, q) not in crops) or '없음'})"],
    ['각인 19종이 가리키는 K-코드', f'{len(set(code_of_imprint.values()))}개 (모두 Kaggle train에 없음: {not set(code_of_imprint.values()) & kaggle_classes})'],
])

**인사이트**
- 00 기록표 453줄이 모두 같은 사진, 같은 사분면의 crop 하나와 이어진다.
    - 1213.png 왼쪽 위(MG)는 RT-DETR이 놓친 알약이라, 1-2에서 그려 넣은 뒤에 이어진다.
- 각인 19종은 K-코드 18개를 가리킨다. 213과 30은 같은 알약(K-018110)의 앞뒤이다.
    - 18개는 모두 Kaggle train에 없고, `pill_detection.datasets.TEST_ONLY_CLASSES`와 같다. (셀이 확인한다)

### 2-2. 자동 클래스 정하기
- 자동 클래스는 아래 순서로 정한다.
    - review_table: 2-1에서 기록표와 이어진 crop. 기록표 각인의 K-코드이다.
    - known_combo: train과 같은 조합인 test 사진의 crop. 그 조합 파일명의 K-코드 중 하나이다.
        - test 파일 번호와 대회 train image id는 한 체계이고, (번호 − 1) ÷ 3의 몫이 같으면 같은 조합을 다른 각도로 찍은 사진이다. (00에서 확인)
        - 같은 조합이어도 각도가 다른 사진은 배치가 돌아가 있다. (예: test 13.png는 train `K-003351-016232-033880_..._75_...`를 180° 돌린 모습이다)
        - 그래서 사분면으로 잇지 않는다. 예측 클래스가 조합의 K-코드이면 그대로 두고, 아니면 다른 crop이 쓰지 않은 K-코드가 하나 남을 때 그 K-코드로 둔다.
    - prediction: 나머지 crop. RT-DETR 예측 클래스이다.

In [ ]:
# test 파일 번호와 대회 train image id는 한 체계이고, (번호 - 1) // 3이 같으면 같은 조합의 다른 각도 사진이다
combo_group = {}  # (image id - 1) // 3 → 그 묶음의 Kaggle train 조합
for name, image in train_images.items():
    if image['source'] == 'kaggle':
        combo_group.setdefault((image['image_id'] - 1) // 3, image['combo'])


def known_classes(f):
    """train과 같은 조합인 test 사진이면 그 조합의 클래스 집합(파일명의 K-코드), 아니면 None."""
    combo = combo_group.get((int(f.split('.')[0]) - 1) // 3)
    return None if combo is None else {int(code) for code in combo.split('-')[1:]}


known_files = [f for f in test_files if known_classes(f) is not None]
for f in known_files:
    # 같은 조합이어도 각도가 다른 사진은 배치가 돌아가 있어서(2-2 소개), 사분면이 아니라 클래스 집합으로 잇는다
    keys = [(f, q) for q in QUADRANTS if (f, q) in crops]
    left = known_classes(f) - {crops[key]['pred'] for key in keys}
    for key in keys:
        crop = crops[key]
        if crop['pred'] in known_classes(f):
            crop['auto'] = crop['pred']
        else:  # 예측이 조합 밖이면, 다른 crop의 예측이 쓰지 않은 클래스가 하나 남을 때만 그 클래스로 둔다
            crop['auto'] = next(iter(left)) if len(left) == 1 and sum(crops[k]['pred'] not in known_classes(f) for k in keys) == 1 else crop['pred']
        crop['source'] = 'known_combo'
for (f, q), crop in crops.items():
    if (f, q) in review_of:
        crop['auto'], crop['source'] = code_of_imprint[review_of[(f, q)]['imprint']], 'review_table'
    elif 'source' not in crop:
        crop['auto'], crop['source'] = crop['pred'], 'prediction'
by_source = collections.defaultdict(list)
for crop in crops.values():
    by_source[crop['source']].append(crop)
show_table(['자동 클래스의 근거', 'crop 수', '클래스 수', '뜻'], [
    ['review_table', len(by_source['review_table']), len({c['auto'] for c in by_source['review_table']}), '00 기록표에 적은 train에 없는 알약. 각인으로 찾은 K-코드'],
    ['known_combo', len(by_source['known_combo']), len({c['auto'] for c in by_source['known_combo']}), f'train과 같은 조합인 test 사진 {len(known_files)}장. 조합 파일명의 K-코드'],
    ['prediction', len(by_source['prediction']), len({c['auto'] for c in by_source['prediction'] if c['auto']}), 'RT-DETR 1차 예측 클래스 (그려 넣은 box는 빈칸)'],
    ['**합계**', f"**{len(crops):,}**", f"**{len({c['auto'] for c in crops.values() if c['auto']})}**", ''],
])

**인사이트**
- review_table 453개(클래스 18개), known_combo 330개(클래스 21개), prediction 2,446개(클래스 52개)이다.
- prediction 중 18개는 1-2에서 그려 넣은 box라 자동 클래스가 없다. 3-2에서 각인으로 정한다.

### 2-3. 예측 클래스의 정확도
- known_combo crop은 조합의 K-코드를 알므로, RT-DETR 예측 클래스가 그 안에 드는지로 정확도를 잰다.
    - prediction crop은 Kaggle 클래스 알약이라는 점에서 known_combo crop과 같다. 이 정확도로 prediction crop이 얼마나 맞을지 가늠한다.

In [ ]:
known_crops = [(key, crop) for key, crop in crops.items() if crop['source'] == 'known_combo' and crop['pred'] is not None]
drawn = [(key, crop) for key, crop in crops.items() if crop['source'] == 'known_combo' and crop['pred'] is None]
wrong = [(key, crop) for key, crop in known_crops if crop['pred'] not in known_classes(key[0])]
code_text = lambda code: f'K-{code:06d}' if code else '정하지 못함'  # noqa: E731
show_table(['항목', '값'], [
    ['train과 같은 조합인 test 사진', f'{len(known_files)}장, RT-DETR crop {len(known_crops)}개'],
    ['RT-DETR 예측 클래스가 조합의 클래스인 crop', f'{len(known_crops) - len(wrong)}개 ({(len(known_crops) - len(wrong)) / len(known_crops):.1%})'],
    ['예측이 조합 밖인 crop (사진 사분면: 예측 → 남은 클래스)',
     ', '.join(f"{f} {q}: K-{c['pred']:06d} → {code_text(c['auto'] if c['auto'] in known_classes(f) else None)}" for (f, q), c in wrong) or '없음'],
    ['1-2에서 그려 넣은 crop (사진 사분면: 남은 클래스)',
     ', '.join(f"{f} {q}: {code_text(c['auto'])}" for (f, q), c in sorted(drawn, key=lambda kv: int(kv[0][0].split('.')[0]))) or '없음'],
])

**인사이트**
- known_combo의 RT-DETR crop 325개는 모두(100.0%) 예측 클래스가 조합의 K-코드 중 하나이다.
    - 이 정확도는 예측이 조합 안에 드는지만 본다. 조합 안의 두 알약을 서로 바꿔 예측해도 맞은 것으로 센다.
- 그려 넣은 crop 5개는 조합에서 남은 K-코드로 정해진다: K-016688 2개(56, 380), K-020014 1개(275), K-032310 2개(650, 1433)

**결론**
- RT-DETR은 Kaggle train 클래스 알약의 클래스를 대체로 맞힌다. prediction crop도 예측 클래스로 묶어서 검수한다.
- 3-1 시트는 score가 낮은 crop부터 놓아, 틀렸을 가능성이 큰 crop을 먼저 본다.

## 3. 클래스별 crop 검수

### 3-1. 클래스별 crop 시트
- 자동 클래스마다 시트 하나를 `runs/eda_raewoo/crops_02/3-1_K-<코드>.png`로 저장한다.
    - 첫 줄(파란 테두리)은 학습 데이터에서 seed를 고정해 고른 그 클래스의 crop 8개이다. Kaggle 사진을 먼저 쓰고, 모자라면 AI Hub에서 채운다.
    - 다음 줄부터 test crop이다. score가 낮은 crop부터 놓는다.
    - crop 아래 글자: test 사진 번호, 사분면(좌위, 우아래 등), 근거(R = review_table, K = known_combo, P = prediction), score
- 시트를 하나씩 열어 기준 crop과 색, 모양, 각인이 다른 test crop을 찾는다.
- 아래 표는 시트 목록이다. crop이 많은 클래스부터 놓는다.

In [ ]:
def reference_crops(code, n=8):
    """학습 데이터에서 그 클래스의 crop n개를 고른다. Kaggle 사진을 먼저 쓰고 모자라면 AI Hub에서 채운다. (seed 고정)"""
    objects = [(image, box) for image in train_images.values() for c, box, _ in image['objects'] if c == code]
    kaggle = [o for o in objects if o[0]['source'] == 'kaggle']
    aihub = [o for o in objects if o[0]['source'] == 'aihub']
    pick = lambda items, k: [items[i] for i in sorted(np.random.default_rng(code).choice(len(items), min(k, len(items)), replace=False))]  # noqa: E731
    chosen = pick(kaggle, n) if kaggle else []
    chosen += pick(aihub, n - len(chosen)) if aihub and len(chosen) < n else []
    return [crop_square(Image.open(DATA_DIR / IMAGE_DIRS[image['source']] / image['file_name']).convert('RGB'), box) for image, box in chosen]


SOURCE_MARK = {'review_table': 'R', 'known_combo': 'K', 'prediction': 'P'}
TILE, COLS = 128, 12
font = ImageFont.truetype('/System/Library/Fonts/Supplemental/AppleGothic.ttf', 13) if Path('/System/Library/Fonts/Supplemental/AppleGothic.ttf').exists() else ImageFont.load_default()


def write_sheet(code, items, path):
    """한 클래스의 crop 시트: 첫 줄은 학습 데이터 crop(파란 테두리), 다음 줄부터 test crop. 글자는 '사진 사분면 근거 score'."""
    refs = reference_crops(code) if code is not None else []
    rows_n = 1 + int(np.ceil(len(items) / COLS))
    sheet = Image.new('RGB', (COLS * TILE, rows_n * (TILE + 18) + 30), 'white')
    draw = ImageDraw.Draw(sheet)
    info = class_info.get(code, {})
    title = (f"K-{code:06d} {class_names.get(code, '')} | 각인 {info.get('print_front', '')}/{info.get('print_back', '')} | "
             f"{info.get('color_class1', '')} {info.get('drug_shape', '')}" if code is not None else '자동 클래스 없음 (1-2에서 그려 넣은 box)')
    draw.text((4, 6), f'{title} | test crop {len(items)}개', fill='black', font=font)
    for i, tile in enumerate(refs):
        sheet.paste(tile, (i * TILE, 30))
        draw.rectangle([i * TILE, 30, (i + 1) * TILE - 1, 30 + TILE - 1], outline='blue', width=3)
    for i, ((f, q), crop) in enumerate(items):
        x, y = (i % COLS) * TILE, 30 + (1 + i // COLS) * (TILE + 18)
        sheet.paste(crop_square(Image.open(TEST_IMG_DIR / f).convert('RGB'), crop['box']), (x, y))
        score = '-' if crop['score'] is None else f"{crop['score']:.2f}"
        draw.text((x + 2, y + TILE), f"{f.split('.')[0]} {q.replace('왼쪽', '좌').replace('오른쪽', '우').replace(' ', '')} {SOURCE_MARK[crop['source']]} {score}",
                  fill='black', font=font)
    sheet.save(path)


groups = collections.defaultdict(list)  # 자동 클래스 → [(key, crop)] (score가 낮은 crop부터)
for key, crop in sorted(crops.items(), key=lambda kv: (kv[1]['score'] is not None, kv[1]['score'] or 0)):
    groups[crop['auto']].append((key, crop))
for code, items in groups.items():
    write_sheet(code, items, SHEET_DIR / (f'3-1_K-{code:06d}.png' if code is not None else '3-1_no_class.png'))
order = sorted((c for c in groups if c is not None), key=lambda c: -len(groups[c]))
show_table(['자동 클래스', '제품명', '그룹', 'test crop 수', 'R / K / P', '최저 score'], [
    [f'K-{c:06d}', class_names.get(c, ''), 'kaggle' if c in kaggle_classes else 'test_only', len(groups[c]),
     ' / '.join(str(sum(cr['source'] == s for _, cr in groups[c])) for s in SOURCE_MARK),
     min((cr['score'] for _, cr in groups[c] if cr['score'] is not None), default='-')] for c in order])

**인사이트**
- 시트 75장(자동 클래스 74개 + 자동 클래스가 없는 그려 넣은 box)의 crop 3,229개를 모두 봤다.
- 가장 많은 K-003483 369개는 모두 초록 장방형(SK / G40)으로 기준 crop과 같다.
- 기준과 다른 알약은 Kaggle train 클래스 시트 10장에서 64개 나왔다. (3-2 표)
    - 모두 RT-DETR이 Kaggle train 클래스끼리 헷갈린 것이다. 00 기록표에 없는 새 알약은 없었다.
    - 틀린 crop의 score는 0.37~0.87이다. 맞은 prediction crop은 95%가 0.77 이상이다.
- test 전용 클래스 18개의 시트(review_table)는 모두 기준과 같다.

### 3-2. 검수 결과
- 3-1에서 자동 클래스와 다른 알약으로 본 crop은 `REVIEW_FIXES`에 최종 클래스와 근거를 적는다.
    - 각인을 읽을 수 있으면 2-1의 `match_imprint`로 학습 데이터 클래스를 찾고, 그 클래스의 crop과 나란히 놓고 확인한다.
    - 학습 데이터 어디에도 없는 알약은 최종 클래스를 비우고(학습할 수 없는 알약), 메모에 각인과 모양을 적는다.
- 아래 표는 최종 클래스가 자동 클래스와 다른 crop이다.

In [ ]:
QUAD_CODE = {'LU': '왼쪽 위', 'RU': '오른쪽 위', 'LD': '왼쪽 아래', 'RD': '오른쪽 아래'}  # 3-1 시트 글자의 사분면 약자
# 3-1 시트에서 자동 클래스와 다른 알약으로 본 crop: (자동 클래스, 최종 클래스, 보이는 알약) → 시트 글자 '사진 번호 + 사분면 약자'
# 모두 RT-DETR이 Kaggle train 클래스끼리 헷갈린 것이다. crop을 크게 확대해 각인을 읽고, 최종 클래스의 학습 데이터 crop과 비교했다
SHEET_FIXES = {
    (1900, 12247, '분홍 장방형 A-008 10'): ['73RD', '1439RD', '1201RD'],
    (16551, 30308, '복숭아색 타원형 D2/850'): ['277LU', '701LU', '700LU'],
    (22347, 20238, '연어색 원형 75 (모서리에 점)'): ['788RU', '465RU', '799RU', '387LD', '464RU', '945LD', '1138RU', '1140RU', '8RU',
                                               '1463RU', '552RU', '9RU', '551LD', '1462LD', '463LD'],
    (35206, 27993, '하양 장방형 ZLT-100'): ['1096RD', '752RD'],
    (34597, 30308, '복숭아색 타원형 D2/850'): ['702RD', '278RD'],
    (29345, 24850, '주황 타원형 Noltec 10'): ['1184RD', '571LU', '1185LU', '573LU', '1012LD', '3LD', '1192RU', '1LD', '1014LD', '581RU', '1013RU'],
    (27926, 30308, '복숭아색 타원형 D2/850 또는 회사 로고'): ['528LU', '1293LU', '527LU', '526RD', '904LU', '906RD', '176RD', '175LU', '560LU'],
    (29451, 36637, '연노랑 타원형 R5'): ['328LD', '115LD', '919LD'],
    (33208, 24850, '주황 타원형 Noltec 10'): ['70RU', '572RD', '986RU', '985LD', '1194LD', '72LD'],
    (27993, 36637, '연노랑 타원형 R5'): ['277RU', '1476RU', '920RU'],
    (27993, 16551, '하양 타원형 DGTH'): ['409LD', '1097RU', '933LD', '932RU', '931RU'],
    (27993, 12247, '분홍 장방형 A-008 10'): ['1438LU', '75RD'],
}
REVIEW_FIXES = {}  # (사진, 사분면) → (최종 클래스, 메모). 최종 클래스가 None이면 학습할 수 없는 알약 (이번 검수에서는 없다)
for (auto, final, seen), labels in SHEET_FIXES.items():
    for label in labels:
        key = (f'{label[:-2]}.png', QUAD_CODE[label[-2:]])
        assert crops[key]['auto'] == auto, f'{key}의 자동 클래스가 K-{auto:06d}이 아니다'
        REVIEW_FIXES[key] = (final, f'3-1 시트 K-{auto:06d}에서 다른 알약 ({seen})')
# 1-2에서 그려 넣었고 기록표·조합으로 클래스를 모르는 알약: 보이는 각인을 match_imprint로 찾은 클래스
MISSED_IMPRINT = {'DGTH': 'DGTH', '515': '515', '81': '81', '5010': '5010', '십자 분할선': '십자분할선'}
for key, (_, seen) in MISSED_PILLS.items():
    if crops[key]['auto'] is None:
        word = next(w for w in MISSED_IMPRINT if w in seen)
        hits = match_imprint(MISSED_IMPRINT[word])
        assert len(hits) == 1, f'{key}의 각인 {word}로 찾은 클래스가 하나가 아니다: {hits}'
        REVIEW_FIXES[key] = (hits[0], f'RT-DETR이 놓친 알약 ({seen}). 각인으로 찾음')
# 3-1 시트의 crop을 모두 눈으로 확인했다 (checked = O)
REVIEWED = True
for key, crop in crops.items():
    crop['final'], crop['note'] = REVIEW_FIXES.get(key, (crop['auto'], ''))
changed = collections.Counter((c['auto'], c['final']) for c in crops.values() if c['final'] != c['auto'])
show_table(['자동 클래스', '최종 클래스', 'crop 수', '근거'], [
    [f'K-{auto:06d}' if auto else '- (그려 넣은 box)', f'K-{final:06d} {class_names[final]}' if final else '학습할 수 없음', n,
     next(c['note'] for c in crops.values() if (c['auto'], c['final']) == (auto, final))]
    for (auto, final), n in sorted(changed.items(), key=lambda kv: (-kv[1], kv[0][0] or 0))]
    + [['**합계**', '', f'**{sum(changed.values())}**', '']])

**인사이트**
- 최종 클래스를 자동 클래스와 다르게 정한 crop은 82개이다.
    - 3-1 시트에서 찾은 다른 알약 64개: 가장 많은 것은 K-022347(자누비아정)로 예측된 K-020238(플라빅스정) 15개이다.
    - 1-2에서 그려 넣었고 기록표나 조합으로 클래스를 모르는 알약 18개: 각인으로 찾았다.
- 학습 데이터 어디에도 없는 알약(학습할 수 없는 알약)은 없다.

**결론**
- 3-2 표의 최종 클래스를 검수표의 `final_category_id`로 쓴다.
- RT-DETR 1차 예측만으로 세었다면, crop 3,229개 중 82개(2.5%)가 틀린 클래스이거나 빠진 알약이었다.

- 3-2에서 최종 클래스를 정한 crop을 그 클래스의 학습 데이터 crop 3개와 나란히 놓고, 색, 모양, 각인이 같은지 확인한다. (01 5-2 방식)

In [ ]:
groups_fixed = collections.defaultdict(list)  # (자동 클래스, 최종 클래스) → crop 키
for key, (final, _) in sorted(REVIEW_FIXES.items(), key=lambda kv: int(kv[0][0].split('.')[0])):
    groups_fixed[(crops[key]['auto'], final)].append(key)
cols = 5  # test crop 2개 + 최종 클래스의 학습 데이터 crop 3개
fig, axes = plt.subplots(len(groups_fixed), cols, figsize=(cols * 1.8, len(groups_fixed) * 2.0), layout='constrained', squeeze=False)
for row, ((auto, final), keys) in zip(axes, sorted(groups_fixed.items(), key=lambda kv: -len(kv[1]))):
    tiles = [crop_square(Image.open(TEST_IMG_DIR / f).convert('RGB'), crops[(f, q)]['box']) for f, q in keys[:2]]
    tiles += [None] * (2 - len(tiles)) + reference_crops(final, n=3)
    for ax, tile in zip(row, tiles):
        if tile is not None:
            ax.imshow(tile)
    row[0].set_title(f"test {keys[0][0]} {keys[0][1]}\n(자동 {f'K-{auto:06d}' if auto else '없음'}, {len(keys)}개 중)", fontsize=7)
    row[2].set_title(f'학습 데이터 K-{final:06d}', fontsize=8)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(f'3-2에서 최종 클래스를 바꾼 crop {len(REVIEW_FIXES)}개의 묶음 {len(groups_fixed)}개\n(왼쪽 2개 = test crop, 오른쪽 3개 = 최종 클래스의 학습 데이터 crop)', fontsize=11)
save_fig(fig, '3-2_review_compare')
plt.show()

**인사이트**
- 17개 묶음 모두 test crop이 최종 클래스의 학습 데이터 crop과 색, 모양, 각인이 같다.
    - 예: K-020238은 연어색 원형에 모서리 점이 있는 75이다. 자동 클래스였던 K-022347(주황 원형 112)과 색과 각인이 다르다.

## 4. 결과

### 4-1. 클래스별 test 객체 수
- 3-2의 최종 클래스로 클래스별 test 객체 수를 센다. 2차 val의 클래스별 목표 객체 수는 이 비율로 정한다. (`pill_detection.datasets.select_val`)

In [ ]:
final_counts = collections.Counter(c['final'] for c in crops.values() if c['final'])
order = sorted(final_counts, key=lambda c: (-final_counts[c], c))
fig, ax = plt.subplots(figsize=(24, 7), layout='constrained')
colors = ['tab:blue' if c in kaggle_classes else 'tab:orange' for c in order]
bars = ax.bar([f'K-{c:06d}' for c in order], [final_counts[c] for c in order], color=colors)
ax.bar_label(bars, fontsize=7)
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.set_ylabel('test 객체 수 (개)')
ax.legend(handles=[matplotlib.patches.Patch(color='tab:blue', label=f'Kaggle train 클래스 {sum(c in kaggle_classes for c in order)}개'),
                   matplotlib.patches.Patch(color='tab:orange', label=f'test 전용 클래스 {sum(c not in kaggle_classes for c in order)}개')])
ax.set_title(f'test 842장의 클래스별 객체 수 (검수한 crop {sum(final_counts.values()):,}개, 클래스 {len(final_counts)}개)\n'
             f'중앙값 {np.median(list(final_counts.values())):g}개, 최소 {min(final_counts.values())}개, 최대 {max(final_counts.values())}개')
save_fig(fig, '4-1_test_class_freq')
plt.show()

**인사이트**
- 대회 클래스 74개가 모두 test에 나온다. 클래스별 객체 수는 3~369개, 중앙값 27개이다.
- K-003483(기넥신에프정)이 369개(전체 crop의 11.4%)로 가장 많다. Kaggle train에는 45개뿐인 클래스이다. (01 4-1)
- 그다음은 K-003351 139개, K-002483 137개, K-001900 122개, K-016551 107개이다.
- test 전용 클래스 18개(주황 막대)는 모두 453개(14.0%)이다.
    - K-027653(세비카정) 73개가 가장 많고, K-010224(넥시움정) 3개가 가장 적다.

### 4-2. 대회 클래스 확정
- 대회 클래스 = Kaggle train 클래스 + 최종 클래스 중 Kaggle train에 없는 클래스(test 전용)
- 00 5-3의 test 전용 18종(`pill_detection.datasets.TEST_ONLY_CLASSES`)과 비교한다.

In [ ]:
test_only = sorted(set(final_counts) - kaggle_classes)
new_classes = sorted(set(test_only) - set(TEST_ONLY_CLASSES))
missing_kaggle = sorted(kaggle_classes - set(final_counts))
unlearnable = [(key, c) for key, c in crops.items() if c['final'] is None]
show_table(['항목', '값'], [
    ['대회 클래스 (Kaggle train + test 전용)', f'{len(kaggle_classes | set(test_only))}개 = {len(kaggle_classes)} + {len(test_only)}'],
    ['00 5-3의 test 전용 18종 밖에서 새로 찾은 클래스', ', '.join(f'K-{c:06d} {class_names.get(c, "")} ({final_counts[c]}개)' for c in new_classes) or '없음'],
    ['test에 나오지 않는 Kaggle train 클래스', ', '.join(f'K-{c:06d}' for c in missing_kaggle) or '없음'],
    ['학습할 수 없는 알약 (학습 데이터 어디에도 없음)', f'{len(unlearnable)}개' + (f" ({', '.join(f'{f} {q}' for (f, q), _ in unlearnable)})" if unlearnable else '')],
])

**인사이트**
- 대회 클래스는 74개 = Kaggle train 56 + test 전용 18이다.
- 00 5-3의 18종 밖에서 새로 찾은 클래스는 없다.
- Kaggle train 클래스 56개는 모두 test에 나온다.
- 학습할 수 없는 알약은 0개이다.

**결론**
- 대회 클래스 74개를 확정한다. `pill_detection.datasets.TEST_ONLY_CLASSES`(18개)를 그대로 쓴다.
    - v3(대회 클래스가 든 이미지)와 v4(대회 클래스 밖 → others)의 정의도 바뀌지 않는다.
- 2차 val의 클래스별 목표 객체 수는 4-1의 test 객체 수 비율로 정한다. `scripts/make_dataset_lists.py`가 아래에서 저장하는 검수표를 읽는다.

- 검수표를 `notebooks/eda/02_raewoo_test_crops.csv`에 저장한다. 사진 번호, 사분면(왼쪽 위, 오른쪽 위, 왼쪽 아래, 오른쪽 아래) 순서이다.
    - `bbox`: crop한 box `[x, y, w, h]`, 원본 픽셀
    - `score`, `pred_category_id`: RT-DETR 1차 예측. 1-2에서 그려 넣은 box는 빈칸이다.
    - `auto_category_id`, `auto_source`: 2-2의 자동 클래스와 근거
    - `final_category_id`: 3-2의 최종 클래스. 학습할 수 없는 알약은 빈칸이다.
    - `checked`: 3-1 시트에서 눈으로 확인했으면 O
    - `note`: 3-2에서 클래스를 바꾼 근거

In [ ]:
with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, lineterminator='\n')
    writer.writerow(['crop_id', 'test_file', 'bbox', 'position', 'score', 'pred_category_id', 'auto_category_id', 'auto_source',
                     'final_category_id', 'checked', 'note'])
    for crop_id, ((f, q), c) in enumerate(sorted(crops.items(), key=lambda kv: (int(kv[0][0].split('.')[0]), QUADRANTS.index(kv[0][1]))), 1):
        writer.writerow([crop_id, f, json.dumps([round(v, 1) for v in c['box']]), q, '' if c['score'] is None else f"{c['score']:.4f}",
                         c['pred'] or '', c['auto'] or '', c['source'], c['final'] or '', 'O' if REVIEWED else '', c['note']])
show_table(['저장한 파일', '줄 수', '열'], [[OUT_CSV.relative_to(PROJECT_ROOT).as_posix(), f'{len(crops):,}', 'crop_id, test_file, bbox, position, score, pred_category_id, auto_category_id, auto_source, final_category_id, checked, note']])

## 5. EDA 결론
- 1~4번의 결과를 항목별 결정 사항으로 정리한다. 괄호 안 번호는 근거가 된 셀의 번호이다.

| 항목 | 결정 | 근거 |
| --- | --- | --- |
| crop 규칙 | 클래스 구분 없는 NMS(IoU 0.5) + 들어간 비율 0.6 이상 버림 + 가장자리 3픽셀 안 box 제외, 최소 score 0.1 | 가장자리 box를 빼면 0.1~0.3에서 사진당 crop 수가 같고, 5개 이상인 사진이 0장 (1-1) |
| 놓친 알약 | 24개를 눈으로 읽은 box로 넣는다 (`MISSED_PILLS`) | 빈 사분면 163곳 중 24곳에 알약이 보였다. 대부분 큰 타원형 알약 (1-2) |
| 자동 클래스 | 기록표 각인 → train과 같은 조합의 K-코드 → RT-DETR 예측 순서 | 기록표 453줄이 모두 crop과 이어지고(2-1), 조합 사진에서 예측이 100% 조합 안 (2-3) |
| 검수 | crop 3,229개를 모두 보고 82개의 최종 클래스를 바꾼다 | Kaggle 클래스끼리 헷갈린 64개 + 그려 넣은 알약 18개. 새 알약 없음 (3-1, 3-2) |
| 대회 클래스 | 74개 = Kaggle train 56 + test 전용 18 (`TEST_ONLY_CLASSES` 그대로) | 18종 밖 새 클래스 0개, 학습할 수 없는 알약 0개 (4-2) |
| 2차 val의 test 빈도 | `02_raewoo_test_crops.csv`의 `final_category_id`로 센 클래스별 객체 수 | 74개 클래스 모두 3~369개, test 전용 14.0% (4-1) |

**추가 실험 후보**
- 2차 모델의 val 클래스별 AP에서 K-016551, K-034597, K-025367, K-031885가 낮으면, 큰 알약을 놓치는 원인(입력 크기, box 크기 분포)을 확인한다.
    - 1차 RT-DETR은 test에서 이 4개 클래스의 알약 17개를 아예 검출하지 못했다. (1-2)
- 2차 모델의 val에서 K-020238과 K-022347의 혼동을 확인한다.
    - 1차 RT-DETR은 test에서 K-020238 15개를 K-022347로 예측했다. (3-2)